In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_DIR = Path("..").resolve()

PROC_DIR = PROJECT_DIR / "Data" / "Processed"
OUT_DIR  = PROJECT_DIR / "Outputs"
SUB_DIR  = OUT_DIR / "Submissions"

SUB_DIR.mkdir(parents=True, exist_ok=True)

print("PROC_DIR:", PROC_DIR)
print("SUB_DIR :", SUB_DIR)

PROC_DIR: /Users/linda/RiceDatathon_2026_Finance/Data/Processed
SUB_DIR : /Users/linda/RiceDatathon_2026_Finance/Outputs/Submissions


In [2]:
panel = pd.read_csv(PROC_DIR / "panel_with_amenity_scores.csv")
scoring = pd.read_csv(PROC_DIR / "scoring_with_amenity_scores.csv")

print("panel  :", panel.shape)
print("scoring:", scoring.shape)

if "time_window_tag" not in panel.columns or "time_window_tag" not in scoring.columns:
    raise RuntimeError("time_window_tag missing in panel or scoring (needed for dual-model).")

print("panel time_window_tag:\n", panel["time_window_tag"].value_counts(dropna=False))
print("scoring time_window_tag:\n", scoring["time_window_tag"].value_counts(dropna=False))

panel  : (38941, 123)
scoring: (8997, 123)
panel time_window_tag:
 time_window_tag
pre     19484
post    19457
Name: count, dtype: int64
scoring time_window_tag:
 time_window_tag
post    4512
pre     4485
Name: count, dtype: int64


In [3]:
BASE_FEATURES = [
    "daily_needs_score",
    "daily_needs_score_equal",
    "lifestyle_score",
    "family_support_score",
    "drivetime_min",
    "numunits",
    "areaperunit",
    "yearbuilt",
    "year_renov",
    "ownrent_avg_rent",
    "ownrent_avg_mortgage",
    "ownrent_spread_abs",
    "ownrent_spread_pct",
    "supply_growth_pct",
    "supply_new_units",
    "supply_baseline_units",
]

FEATURES = [c for c in BASE_FEATURES if c in panel.columns and c in scoring.columns]
if len(FEATURES) < 5:
    raise RuntimeError(f"Too few features found: {FEATURES}")

ID_CANDIDATES = ["ubid", "property_id"]
GROUP_COL = next((c for c in ID_CANDIDATES if c in panel.columns), None)

if GROUP_COL is None:
    stable_candidates = ["latitude", "longitude", "lat", "lon", "yearbuilt", "numunits"]
    stable = [c for c in stable_candidates if c in panel.columns]
    if len(stable) == 0:
        raise RuntimeError("No ID columns and no stable columns to construct group key.")
    panel["_group_key"] = panel[stable].astype(str).agg("_".join, axis=1)
    scoring["_group_key"] = scoring[stable].astype(str).agg("_".join, axis=1)
    GROUP_COL = "_group_key"

print("n_features:", len(FEATURES))
print("Using group column:", GROUP_COL)

n_features: 16
Using group column: ubid


In [4]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import mean_squared_error
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from lightgbm import LGBMRegressor

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

In [5]:
def train_one_window(df_train: pd.DataFrame, target_source: str, name: str):
    df = df_train.copy()
    df["target"] = pd.to_numeric(df[target_source], errors="coerce")
    df = df[df["target"].notna()].copy()

    X = df[FEATURES].copy()
    y = df["target"].copy()
    groups = df[GROUP_COL].astype(str).values

    model = LGBMRegressor(
        n_estimators=3500,
        learning_rate=0.02,
        num_leaves=63,
        min_child_samples=30,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=1.0,
        random_state=42,
    )

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", model),
    ])

    gkf = GroupKFold(n_splits=5)
    rmses = []

    for fold, (tr_idx, va_idx) in enumerate(gkf.split(X, y, groups=groups), start=1):
        X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

        pipe.fit(X_tr, y_tr)
        pred_va = pipe.predict(X_va)
        fold_rmse = rmse(y_va, pred_va)
        rmses.append(fold_rmse)
        print(f"[{name}] Fold {fold}: RMSE = {fold_rmse:.5f}")

    print(f"[{name}] CV RMSE mean: {float(np.mean(rmses)):.5f}")
    print(f"[{name}] CV RMSE std : {float(np.std(rmses)):.5f}")

    pipe.fit(X, y)
    return pipe

In [6]:
pre_train = panel[panel["time_window_tag"] == "pre"].copy()
if pre_train.empty:
    raise RuntimeError("No pre rows found in panel (time_window_tag == 'pre').")

PRE_TARGET = "revpar_growth_2015_2020_pct"
if PRE_TARGET not in pre_train.columns:
    raise RuntimeError(f"Missing pre target column: {PRE_TARGET}")

pre_model = train_one_window(pre_train, PRE_TARGET, "PRE")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000454 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3426
[LightGBM] [Info] Number of data points in the train set: 15587, number of used features: 16
[LightGBM] [Info] Start training from score 0.216620


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[PRE] Fold 1: RMSE = 0.15404
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000356 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3427
[LightGBM] [Info] Number of data points in the train set: 15587, number of used features: 16
[LightGBM] [Info] Start training from score 0.218346


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[PRE] Fold 2: RMSE = 0.18979
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000373 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3426
[LightGBM] [Info] Number of data points in the train set: 15587, number of used features: 16
[LightGBM] [Info] Start training from score 0.216801


...

[PRE] Fold 3: RMSE = 0.16683
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000404 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3424
[LightGBM] [Info] Number of data points in the train set: 15587, number of used features: 16
[LightGBM] [Info] Start training from score 0.218363


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[PRE] Fold 4: RMSE = 0.15742
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000374 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3428
[LightGBM] [Info] Number of data points in the train set: 15588, number of used features: 16
[LightGBM] [Info] Start training from score 0.216953


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[PRE] Fold 5: RMSE = 0.18720
[PRE] CV RMSE mean: 0.17106
[PRE] CV RMSE std : 0.01487
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000352 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3431
[LightGBM] [Info] Number of data points in the train set: 19484, number of used features: 16
[LightGBM] [Info] Start training from score 0.217417


In [7]:
post_train = panel[panel["time_window_tag"] == "post"].copy()
if post_train.empty:
    raise RuntimeError("No post rows found in panel (time_window_tag == 'post').")

POST_TARGET = "revpar_growth_2022_2025_pct"
if POST_TARGET not in post_train.columns:
    raise RuntimeError(f"Missing post target column: {POST_TARGET}")

post_model = train_one_window(post_train, POST_TARGET, "POST")

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000353 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3429
[LightGBM] [Info] Number of data points in the train set: 15565, number of used features: 16
[LightGBM] [Info] Start training from score -0.051480


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[POST] Fold 1: RMSE = 0.12796
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000377 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3425
[LightGBM] [Info] Number of data points in the train set: 15565, number of used features: 16
[LightGBM] [Info] Start training from score -0.050974


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[POST] Fold 2: RMSE = 0.11898
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000452 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3430
[LightGBM] [Info] Number of data points in the train set: 15566, number of used features: 16
[LightGBM] [Info] Start training from score -0.052151


...

[POST] Fold 3: RMSE = 0.12233
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000531 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3432
[LightGBM] [Info] Number of data points in the train set: 15566, number of used features: 16
[LightGBM] [Info] Start training from score -0.051570


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[POST] Fold 4: RMSE = 0.12032
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000476 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3427
[LightGBM] [Info] Number of data points in the train set: 15566, number of used features: 16
[LightGBM] [Info] Start training from score -0.052490


/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


[POST] Fold 5: RMSE = 0.13124
[POST] CV RMSE mean: 0.12417
[POST] CV RMSE std : 0.00468
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001257 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3434
[LightGBM] [Info] Number of data points in the train set: 19457, number of used features: 16
[LightGBM] [Info] Start training from score -0.051733


In [8]:
sc = scoring.copy()
X_test = sc[FEATURES].copy()

pred = np.full(len(sc), np.nan, dtype=float)

mask_pre  = sc["time_window_tag"] == "pre"
mask_post = sc["time_window_tag"] == "post"

if mask_pre.any():
    pred[mask_pre.values] = pre_model.predict(X_test.loc[mask_pre])

if mask_post.any():
    pred[mask_post.values] = post_model.predict(X_test.loc[mask_post])

if np.isnan(pred).any():
    missing_tags = sc.loc[np.isnan(pred), "time_window_tag"].value_counts(dropna=False)
    raise RuntimeError(f"Some scoring rows not predicted. Missing tags:\n{missing_tags}")

sc["pred_target_lgbm_dual"] = pred
print(sc["pred_target_lgbm_dual"].describe())

/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/opt/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


count    8997.000000
mean        0.079708
std         0.177610
min        -0.531231
25%        -0.062207
50%         0.046374
75%         0.204050
max         2.517296
Name: pred_target_lgbm_dual, dtype: float64


In [9]:
ID_CANDIDATES = ["row_id", "id", "property_id", "ubid"]
ID_COL = next((c for c in ID_CANDIDATES if c in scoring.columns), None)

if ID_COL is None:
    submission = sc[["pred_target_lgbm_dual"]].copy()
else:
    submission = sc[[ID_COL, "pred_target_lgbm_dual"]].copy()

sub_path = SUB_DIR / "submission_lgbm_dual_pre_post.csv"
submission.to_csv(sub_path, index=False)

print("Saved submission:", sub_path)
print(submission.head())

Saved submission: /Users/linda/RiceDatathon_2026_Finance/Outputs/Submissions/submission_lgbm_dual_pre_post.csv
                       ubid  pred_target_lgbm_dual
0  76X6WHJM+WGJ-18-14-18-15               0.436546
1  865523H8+8QC-18-14-18-14              -0.065655
2  865538F3+9FQ-18-14-18-15              -0.119850
3  865QRR26+P69-18-14-18-15              -0.061853
4  862476FP+H6F-18-14-18-15              -0.045066


In [10]:
def get_gain_importance(pipe: Pipeline, feature_names: list[str]) -> pd.DataFrame:
    m = pipe.named_steps["model"]
    gain = m.booster_.feature_importance(importance_type="gain").astype(float)
    gain = gain / (gain.sum() + 1e-12)
    return pd.DataFrame({"feature": feature_names, "gain_norm": gain})

imp_pre = get_gain_importance(pre_model, FEATURES).rename(columns={"gain_norm":"gain_pre"})
imp_post = get_gain_importance(post_model, FEATURES).rename(columns={"gain_norm":"gain_post"})

shift = imp_pre.merge(imp_post, on="feature", how="outer").fillna(0.0)
shift["delta_post_minus_pre"] = shift["gain_post"] - shift["gain_pre"]

shift_path = PROC_DIR / "dual_model_importance_shift.csv"
shift.to_csv(shift_path, index=False)

print("Saved:", shift_path)
shift.sort_values("delta_post_minus_pre", ascending=False).head(10)

Saved: /Users/linda/RiceDatathon_2026_Finance/Data/Processed/dual_model_importance_shift.csv


,feature,gain_pre,gain_post,delta_post_minus_pre
6,numunits,0.094533,0.164392,0.069859
0,areaperunit,0.101762,0.124991,0.023229
10,ownrent_spread_pct,0.035888,0.049464,0.013576
8,ownrent_avg_rent,0.089720,0.101468,0.011749
12,supply_growth_pct,0.048917,0.059792,0.010875
5,lifestyle_score,0.052795,0.062368,0.009574
4,family_support_score,0.074641,0.083121,0.008480
1,daily_needs_score,0.048684,0.047460,-0.001224
9,ownrent_spread_abs,0.036542,0.034573,-0.001970
14,year_renov,0.041468,0.037097,-0.004371
